# Trend words — visual exploration

Runs [`trending.sql`](../trending.sql) against the Azure PostgreSQL database and
plots the top rising n-grams.

**Setup (once):** create an isolated environment for this notebook — it is kept
separate from the `python/` services on purpose.

```bash
cd notebook
uv sync            # or: pip install -r requirements.txt
```

Connection settings are read from the repository root `.env`
(`POSTGRES_HOST`, `POSTGRES_USER`, `POSTGRES_DATABASE`, `POSTGRES_PASSWORD`,
`PGSSLMODE`). For Azure set `PGSSLMODE=require`.

In [ ]:
import os
from pathlib import Path

import pandas as pd
import plotly.express as px
import psycopg
from dotenv import load_dotenv

# Load connection settings from the repository root .env
load_dotenv(Path("..") / ".env")

conninfo = psycopg.conninfo.make_conninfo(
    host=os.environ["POSTGRES_HOST"],
    port=int(os.getenv("POSTGRES_PORT", "5432")),
    user=os.environ["POSTGRES_USER"],
    password=os.environ["POSTGRES_PASSWORD"],
    dbname=os.environ["POSTGRES_DATABASE"],
    sslmode=os.getenv("PGSSLMODE", "require"),  # Azure PG requires SSL
)
print("Connecting to", os.environ["POSTGRES_HOST"])

In [ ]:
# Load the query straight from trending.sql so query and plots stay in sync
sql = (Path("..") / "trending.sql").read_text()

with psycopg.connect(conninfo) as conn:
    df = pd.read_sql(sql, conn)

print(f"{len(df)} rows")
df.head(20)

## Top trends by statistical surprise (z-score)

Bars are colored by how many distinct sources corroborate the phrase — taller
and darker means both surprising and widely reported.

In [ ]:
top = df.head(20).sort_values("z_score")

fig = px.bar(
    top,
    x="z_score",
    y="words",
    orientation="h",
    color="recent_sources",
    color_continuous_scale="Viridis",
    hover_data=["recent_count", "baseline_count", "growth_ratio"],
    title="Top rising n-grams by Poisson z-score",
)
fig.update_layout(height=650, yaxis_title="", xaxis_title="z-score")
fig.show()

## Growth vs. volume

Each bubble is a phrase: x = current volume, y = smoothed growth ratio,
size = distinct sources. Top-right, large bubbles are the strongest,
best-corroborated trends.

In [ ]:
fig = px.scatter(
    df,
    x="recent_count",
    y="growth_ratio",
    size="recent_sources",
    color="z_score",
    color_continuous_scale="Plasma",
    hover_name="words",
    hover_data=["baseline_count", "absolute_growth", "recent_sources"],
    title="Growth ratio vs. recent volume",
)
fig.update_layout(height=600, xaxis_title="recent occurrences", yaxis_title="growth ratio")
fig.show()